# TraderPro - Reinforcement Learning Agent Training

Ce notebook entraîne un agent RL pour le trading Bitcoin.

## Configuration choisie
- **Actions**: Discrètes (BUY / SELL / HOLD)
- **Horizon**: Daily (1 décision par jour)
- **Objectif**: Sharpe Ratio (profit ajusté au risque)
- **Algorithme**: PPO (Proximal Policy Optimization)
- **Data Source**: Twelve Data API (3 ans d'historique)

## Étapes
1. Installation des dépendances
2. Récupération des données via Twelve Data API
3. Création de l'environnement de trading
4. Entraînement de l'agent
5. Évaluation et backtesting
6. Sauvegarde du modèle

---
## 1. Installation des dépendances

In [ ]:
!pip install stable-baselines3[extra] gymnasium pandas numpy requests ta matplotlib --quiet
print("Installation terminée!")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Reinforcement Learning
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO, A2C, DQN
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback

print("Imports OK!")
print(f"Gymnasium version: {gym.__version__}")

---
## 2. Récupération des données Twelve Data API

L'API Binance ne fonctionne pas sur certains serveurs Colab, donc on utilise Twelve Data comme alternative.

In [ ]:
# Configuration Twelve Data API
TWELVEDATA_API_KEY = "7521bc3c3e92455181d590829c90bf9e"

def fetch_twelvedata(symbol="BTC/USD", interval="1day", years=3):
    """
    Récupère les données historiques depuis Twelve Data API.
    
    Args:
        symbol: Paire de trading (BTC/USD)
        interval: Intervalle (1day pour daily)
        years: Nombre d'années d'historique
    
    Returns:
        DataFrame avec OHLCV
    """
    from datetime import datetime, timedelta
    
    # Calculer les dates
    end_date = datetime.now()
    start_date = end_date - timedelta(days=365 * years)
    
    # URL API Twelve Data
    url = "https://api.twelvedata.com/time_series"
    
    params = {
        "symbol": symbol,
        "interval": interval,
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "order": "asc",
        "apikey": TWELVEDATA_API_KEY,
        "outputsize": 5000  # Maximum
    }
    
    print(f"Récupération des données {symbol}...")
    print(f"Période: {start_date.strftime('%Y-%m-%d')} à {end_date.strftime('%Y-%m-%d')}")
    
    response = requests.get(url, params=params)
    data = response.json()
    
    # Vérifier les erreurs
    if "code" in data and data["code"] != 200:
        raise Exception(f"Erreur API: {data.get('message', 'Unknown error')}")
    
    if "values" not in data:
        raise Exception(f"Pas de données reçues. Réponse: {data}")
    
    # Convertir en DataFrame
    df = pd.DataFrame(data["values"])
    
    # Renommer les colonnes
    df = df.rename(columns={
        "datetime": "timestamp",
        "open": "open",
        "high": "high",
        "low": "low",
        "close": "close",
        "volume": "volume"
    })
    
    # Convertir les types
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('timestamp')
    
    for col in ['open', 'high', 'low', 'close', 'volume']:
        df[col] = df[col].astype(float)
    
    # Garder seulement OHLCV
    df = df[['open', 'high', 'low', 'close', 'volume']]
    
    # Trier par date (au cas où)
    df = df.sort_index()
    
    return df

# Récupérer 3 ans de données daily
print("Récupération des données Twelve Data...")
df = fetch_twelvedata("BTC/USD", "1day", years=3)
print(f"Données récupérées: {len(df)} jours")
print(f"Période: {df.index[0]} à {df.index[-1]}")
df.tail()

---
## 3. Feature Engineering

In [ ]:
def create_features(df):
    """
    Crée les features pour l'agent RL.
    
    Features:
    - Returns (rendements)
    - Volatilité
    - RSI
    - MACD
    - Position dans Bollinger Bands
    - Volume ratio
    """
    data = df.copy()
    
    # Returns
    data['returns'] = data['close'].pct_change()
    data['returns_2d'] = data['close'].pct_change(2)
    data['returns_5d'] = data['close'].pct_change(5)
    data['returns_10d'] = data['close'].pct_change(10)
    
    # Volatilité
    data['volatility_7d'] = data['returns'].rolling(7).std()
    data['volatility_14d'] = data['returns'].rolling(14).std()
    
    # RSI
    delta = data['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / (loss + 1e-10)
    data['rsi'] = 100 - (100 / (1 + rs))
    data['rsi_normalized'] = data['rsi'] / 100  # Normaliser entre 0 et 1
    
    # MACD
    ema12 = data['close'].ewm(span=12).mean()
    ema26 = data['close'].ewm(span=26).mean()
    data['macd'] = ema12 - ema26
    data['macd_signal'] = data['macd'].ewm(span=9).mean()
    data['macd_hist'] = data['macd'] - data['macd_signal']
    data['macd_normalized'] = data['macd_hist'] / data['close']  # Normaliser
    
    # Bollinger Bands
    sma20 = data['close'].rolling(20).mean()
    std20 = data['close'].rolling(20).std()
    data['bb_upper'] = sma20 + 2 * std20
    data['bb_lower'] = sma20 - 2 * std20
    data['bb_position'] = (data['close'] - data['bb_lower']) / (data['bb_upper'] - data['bb_lower'] + 1e-10)
    
    # Moving Averages
    data['sma_7'] = data['close'].rolling(7).mean()
    data['sma_25'] = data['close'].rolling(25).mean()
    data['sma_50'] = data['close'].rolling(50).mean()
    data['price_sma7_ratio'] = data['close'] / data['sma_7']
    data['price_sma25_ratio'] = data['close'] / data['sma_25']
    data['sma7_sma25_ratio'] = data['sma_7'] / data['sma_25']
    
    # Volume
    data['volume_sma'] = data['volume'].rolling(20).mean()
    data['volume_ratio'] = data['volume'] / (data['volume_sma'] + 1e-10)
    
    # Range
    data['daily_range'] = (data['high'] - data['low']) / data['close']
    
    # Clean
    data = data.replace([np.inf, -np.inf], np.nan)
    data = data.dropna()
    
    return data

# Créer les features
df_features = create_features(df)
print(f"Features créées: {len(df_features)} lignes")
print(f"Colonnes: {list(df_features.columns)}")

---
## 4. Environnement de Trading (Gymnasium)

In [ ]:
class TradingEnvironment(gym.Env):
    """
    Environnement de trading pour Reinforcement Learning.
    
    Actions:
        0 = HOLD (ne rien faire)
        1 = BUY (acheter)
        2 = SELL (vendre)
    
    State:
        - Features techniques normalisées
        - Position actuelle
        - PnL non réalisé
    
    Reward:
        - Sharpe ratio quotidien
        - Pénalité pour drawdown
        - Pénalité pour frais
    """
    
    metadata = {'render_modes': ['human']}
    
    def __init__(self, df, initial_capital=10000, transaction_fee=0.001, window_size=14):
        super().__init__()
        
        self.df = df.reset_index(drop=True)
        self.initial_capital = initial_capital
        self.transaction_fee = transaction_fee
        self.window_size = window_size
        
        # Features à utiliser pour l'observation
        self.feature_columns = [
            'returns', 'returns_2d', 'returns_5d', 'returns_10d',
            'volatility_7d', 'volatility_14d',
            'rsi_normalized', 'macd_normalized', 'bb_position',
            'price_sma7_ratio', 'price_sma25_ratio', 'sma7_sma25_ratio',
            'volume_ratio', 'daily_range'
        ]
        
        # Espace d'actions: 0=HOLD, 1=BUY, 2=SELL
        self.action_space = spaces.Discrete(3)
        
        # Espace d'observations: features + position + unrealized_pnl
        n_features = len(self.feature_columns) + 2  # +2 pour position et pnl
        self.observation_space = spaces.Box(
            low=-np.inf, 
            high=np.inf, 
            shape=(n_features,), 
            dtype=np.float32
        )
        
        self.reset()
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        self.current_step = self.window_size
        self.capital = self.initial_capital
        self.position = 0  # 0 = pas de position, 1 = long
        self.entry_price = 0
        self.total_profit = 0
        self.trades = []
        self.portfolio_values = [self.initial_capital]
        self.returns_history = []
        self.max_portfolio_value = self.initial_capital
        
        return self._get_observation(), {}
    
    def _get_observation(self):
        """Retourne l'observation actuelle."""
        # Features techniques
        features = self.df.loc[self.current_step, self.feature_columns].values
        
        # Ajouter position et PnL non réalisé
        unrealized_pnl = 0
        if self.position == 1:
            current_price = self.df.loc[self.current_step, 'close']
            unrealized_pnl = (current_price - self.entry_price) / self.entry_price
        
        obs = np.concatenate([
            features,
            [self.position, unrealized_pnl]
        ]).astype(np.float32)
        
        # Remplacer NaN par 0
        obs = np.nan_to_num(obs, nan=0.0, posinf=0.0, neginf=0.0)
        
        return obs
    
    def step(self, action):
        """Exécute une action et retourne (obs, reward, terminated, truncated, info)."""
        current_price = self.df.loc[self.current_step, 'close']
        reward = 0
        trade_profit = 0
        
        # Exécuter l'action
        if action == 1:  # BUY
            if self.position == 0:  # Pas de position -> acheter
                self.position = 1
                self.entry_price = current_price
                # Frais de transaction
                self.capital *= (1 - self.transaction_fee)
                
        elif action == 2:  # SELL
            if self.position == 1:  # Position longue -> vendre
                # Calculer le profit
                trade_profit = (current_price - self.entry_price) / self.entry_price
                self.capital *= (1 + trade_profit) * (1 - self.transaction_fee)
                self.total_profit += trade_profit
                self.trades.append({
                    'entry': self.entry_price,
                    'exit': current_price,
                    'profit': trade_profit
                })
                self.position = 0
                self.entry_price = 0
        
        # Mettre à jour la valeur du portfolio
        if self.position == 1:
            unrealized = (current_price - self.entry_price) / self.entry_price
            portfolio_value = self.capital * (1 + unrealized)
        else:
            portfolio_value = self.capital
        
        self.portfolio_values.append(portfolio_value)
        self.max_portfolio_value = max(self.max_portfolio_value, portfolio_value)
        
        # Calculer le return quotidien
        daily_return = (portfolio_value - self.portfolio_values[-2]) / self.portfolio_values[-2] if len(self.portfolio_values) > 1 else 0
        self.returns_history.append(daily_return)
        
        # === REWARD FUNCTION (Sharpe-based) ===
        
        # 1. Reward basé sur le Sharpe Ratio glissant
        if len(self.returns_history) >= 7:
            recent_returns = np.array(self.returns_history[-7:])
            if np.std(recent_returns) > 0:
                sharpe = np.mean(recent_returns) / np.std(recent_returns) * np.sqrt(252)
                reward += sharpe * 0.1
        
        # 2. Reward direct pour profit réalisé
        reward += trade_profit * 10
        
        # 3. Pénalité pour drawdown
        drawdown = (self.max_portfolio_value - portfolio_value) / self.max_portfolio_value
        if drawdown > 0.1:  # Plus de 10% de drawdown
            reward -= drawdown * 5
        
        # 4. Petite pénalité pour HOLD pendant trop longtemps sans position
        if action == 0 and self.position == 0:
            reward -= 0.001  # Encourage à prendre des positions
        
        # Avancer d'un pas
        self.current_step += 1
        
        # Vérifier si l'épisode est terminé
        terminated = self.current_step >= len(self.df) - 1
        truncated = self.capital <= self.initial_capital * 0.5  # Stop si perte > 50%
        
        info = {
            'capital': self.capital,
            'portfolio_value': portfolio_value,
            'position': self.position,
            'total_trades': len(self.trades),
            'total_profit': self.total_profit,
            'drawdown': drawdown
        }
        
        return self._get_observation(), reward, terminated, truncated, info
    
    def get_metrics(self):
        """Retourne les métriques de performance."""
        if len(self.portfolio_values) < 2:
            return {}
        
        returns = np.diff(self.portfolio_values) / self.portfolio_values[:-1]
        
        total_return = (self.portfolio_values[-1] - self.initial_capital) / self.initial_capital
        sharpe = np.mean(returns) / (np.std(returns) + 1e-10) * np.sqrt(252) if len(returns) > 0 else 0
        max_dd = np.max(np.maximum.accumulate(self.portfolio_values) - self.portfolio_values) / np.maximum.accumulate(self.portfolio_values).max()
        
        winning_trades = [t for t in self.trades if t['profit'] > 0]
        win_rate = len(winning_trades) / len(self.trades) if self.trades else 0
        
        return {
            'total_return': total_return,
            'sharpe_ratio': sharpe,
            'max_drawdown': max_dd,
            'total_trades': len(self.trades),
            'win_rate': win_rate,
            'final_capital': self.portfolio_values[-1]
        }

# Test de l'environnement
print("Test de l'environnement...")
env = TradingEnvironment(df_features)
obs, info = env.reset()
print(f"Observation shape: {obs.shape}")
print(f"Action space: {env.action_space}")
print(f"Observation space: {env.observation_space}")

# Test quelques steps
for i in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"Step {i+1}: Action={['HOLD', 'BUY', 'SELL'][action]}, Reward={reward:.4f}, Capital={info['capital']:.2f}")

---
## 5. Entraînement de l'Agent PPO

In [ ]:
# Callback pour suivre l'entraînement
class TradingCallback(BaseCallback):
    def __init__(self, eval_env, eval_freq=1000, verbose=1):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.best_sharpe = -np.inf
        self.metrics_history = []
    
    def _on_step(self):
        if self.n_calls % self.eval_freq == 0:
            # Évaluer l'agent
            obs, _ = self.eval_env.reset()
            done = False
            while not done:
                action, _ = self.model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = self.eval_env.step(action)
                done = terminated or truncated
            
            metrics = self.eval_env.get_metrics()
            self.metrics_history.append(metrics)
            
            if self.verbose > 0:
                print(f"\nStep {self.n_calls}:")
                print(f"  Return: {metrics.get('total_return', 0)*100:.2f}%")
                print(f"  Sharpe: {metrics.get('sharpe_ratio', 0):.2f}")
                print(f"  Trades: {metrics.get('total_trades', 0)}")
                print(f"  Win Rate: {metrics.get('win_rate', 0)*100:.1f}%")
            
            # Sauvegarder le meilleur modèle
            if metrics.get('sharpe_ratio', 0) > self.best_sharpe:
                self.best_sharpe = metrics.get('sharpe_ratio', 0)
                self.model.save("best_trading_agent")
                if self.verbose > 0:
                    print(f"  New best model saved! Sharpe: {self.best_sharpe:.2f}")
        
        return True

In [ ]:
# Séparer train/test (80/20 temporel)
split_idx = int(len(df_features) * 0.8)
train_df = df_features.iloc[:split_idx].reset_index(drop=True)
test_df = df_features.iloc[split_idx:].reset_index(drop=True)

print(f"Train: {len(train_df)} jours")
print(f"Test: {len(test_df)} jours")

# Créer les environnements
train_env = TradingEnvironment(train_df)
eval_env = TradingEnvironment(test_df)

# Callback
callback = TradingCallback(eval_env, eval_freq=5000, verbose=1)

In [ ]:
# Créer le modèle PPO
model = PPO(
    "MlpPolicy",
    train_env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,  # Encourage l'exploration
    verbose=1,
    tensorboard_log="./tensorboard_logs/"
)

print("Modèle PPO créé!")
print(f"Policy: {model.policy}")

In [ ]:
# ENTRAÎNEMENT
# Ajustez total_timesteps selon vos besoins (plus = meilleur mais plus long)
# 50000 ~ 5-10 min, 200000 ~ 30 min, 500000 ~ 1h+

TOTAL_TIMESTEPS = 100000  # Commencez petit, augmentez si ça marche

print(f"Début de l'entraînement ({TOTAL_TIMESTEPS} timesteps)...")
print("="*50)

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=callback,
    progress_bar=True
)

print("="*50)
print("Entraînement terminé!")

---
## 6. Évaluation et Backtesting

In [ ]:
# Charger le meilleur modèle
best_model = PPO.load("best_trading_agent")
print("Meilleur modèle chargé!")

# Évaluer sur les données de test
test_env = TradingEnvironment(test_df)
obs, _ = test_env.reset()

actions_taken = []
portfolio_history = [test_env.initial_capital]
price_history = [test_df.iloc[test_env.window_size]['close']]

done = False
while not done:
    action, _ = best_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = test_env.step(action)
    done = terminated or truncated
    
    actions_taken.append(['HOLD', 'BUY', 'SELL'][action])
    portfolio_history.append(info['portfolio_value'])
    price_history.append(test_df.iloc[test_env.current_step]['close'] if test_env.current_step < len(test_df) else price_history[-1])

# Métriques finales
metrics = test_env.get_metrics()
print("\n" + "="*50)
print("RÉSULTATS SUR DONNÉES DE TEST")
print("="*50)
print(f"Return Total: {metrics['total_return']*100:.2f}%")
print(f"Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
print(f"Max Drawdown: {metrics['max_drawdown']*100:.2f}%")
print(f"Nombre de Trades: {metrics['total_trades']}")
print(f"Win Rate: {metrics['win_rate']*100:.1f}%")
print(f"Capital Final: ${metrics['final_capital']:,.2f}")

In [ ]:
# Visualisation
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# 1. Portfolio vs Buy & Hold
ax1 = axes[0]
initial_price = price_history[0]
buy_hold = [test_env.initial_capital * (p / initial_price) for p in price_history]

ax1.plot(portfolio_history, label='Agent RL', color='green', linewidth=2)
ax1.plot(buy_hold, label='Buy & Hold', color='blue', linestyle='--', alpha=0.7)
ax1.axhline(y=test_env.initial_capital, color='gray', linestyle=':', alpha=0.5)
ax1.set_title('Performance: Agent RL vs Buy & Hold', fontsize=14)
ax1.set_ylabel('Valeur du Portfolio ($)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Prix BTC avec signaux
ax2 = axes[1]
ax2.plot(price_history, label='Prix BTC', color='orange', alpha=0.7)

# Marquer les BUY/SELL
for i, action in enumerate(actions_taken):
    if action == 'BUY':
        ax2.scatter(i, price_history[i], color='green', marker='^', s=100, zorder=5)
    elif action == 'SELL':
        ax2.scatter(i, price_history[i], color='red', marker='v', s=100, zorder=5)

ax2.set_title('Prix BTC avec Signaux de Trading', fontsize=14)
ax2.set_ylabel('Prix ($)')
ax2.legend(['Prix BTC', 'BUY', 'SELL'])
ax2.grid(True, alpha=0.3)

# 3. Distribution des actions
ax3 = axes[2]
action_counts = pd.Series(actions_taken).value_counts()
colors = {'HOLD': 'gray', 'BUY': 'green', 'SELL': 'red'}
ax3.bar(action_counts.index, action_counts.values, color=[colors[a] for a in action_counts.index])
ax3.set_title('Distribution des Actions', fontsize=14)
ax3.set_ylabel('Nombre')

plt.tight_layout()
plt.savefig('rl_backtest_results.png', dpi=150)
plt.show()

print("\nGraphique sauvegardé: rl_backtest_results.png")

---
## 7. Sauvegarde du Modèle pour TraderPro

In [ ]:
import json
import pickle

# Sauvegarder le modèle final
best_model.save("trading_agent_ppo_final")

# Sauvegarder la configuration
config = {
    "model_type": "PPO",
    "model_version": "1.0.0",
    "trained_at": datetime.now().isoformat(),
    "training_timesteps": TOTAL_TIMESTEPS,
    "feature_columns": [
        'returns', 'returns_2d', 'returns_5d', 'returns_10d',
        'volatility_7d', 'volatility_14d',
        'rsi_normalized', 'macd_normalized', 'bb_position',
        'price_sma7_ratio', 'price_sma25_ratio', 'sma7_sma25_ratio',
        'volume_ratio', 'daily_range'
    ],
    "actions": ["HOLD", "BUY", "SELL"],
    "action_space": "Discrete(3)",
    "initial_capital": 10000,
    "transaction_fee": 0.001,
    "metrics": metrics
}

with open("training_config_rl.json", "w") as f:
    json.dump(config, f, indent=2, default=str)

print("Fichiers sauvegardés:")
print("  - trading_agent_ppo_final.zip (modèle)")
print("  - training_config_rl.json (configuration)")
print("  - rl_backtest_results.png (graphique)")

In [ ]:
# Télécharger les fichiers
from google.colab import files

print("Téléchargement des fichiers...")
files.download('trading_agent_ppo_final.zip')
files.download('training_config_rl.json')
files.download('rl_backtest_results.png')

print("\n" + "="*50)
print("INSTRUCTIONS POUR TRADERPRO")
print("="*50)
print("""
1. Placez les fichiers dans: backend/app/ml/models/
   - trading_agent_ppo_final.zip
   - training_config_rl.json

2. Le service RL chargera automatiquement le modèle

3. Utilisez l'endpoint: GET /api/rl/action
""")

---
## 8. Test du Modèle (Simulation d'une Prédiction)

In [ ]:
# Simuler ce que fera le backend
def predict_action(model, current_data):
    """
    Prédit l'action à prendre basée sur les données actuelles.
    
    Args:
        model: Le modèle PPO entraîné
        current_data: DataFrame avec les dernières données
    
    Returns:
        dict: Action recommandée et métadonnées
    """
    # Créer les features
    features_df = create_features(current_data)
    
    feature_columns = [
        'returns', 'returns_2d', 'returns_5d', 'returns_10d',
        'volatility_7d', 'volatility_14d',
        'rsi_normalized', 'macd_normalized', 'bb_position',
        'price_sma7_ratio', 'price_sma25_ratio', 'sma7_sma25_ratio',
        'volume_ratio', 'daily_range'
    ]
    
    # Dernière observation
    last_features = features_df[feature_columns].iloc[-1].values
    
    # Ajouter position et PnL (0 pour une nouvelle prédiction)
    obs = np.concatenate([last_features, [0, 0]]).astype(np.float32)
    obs = np.nan_to_num(obs, nan=0.0)
    
    # Prédire
    action, _ = model.predict(obs, deterministic=True)
    
    action_names = ["HOLD", "BUY", "SELL"]
    
    return {
        "action": action_names[action],
        "action_code": int(action),
        "current_price": float(current_data['close'].iloc[-1]),
        "timestamp": datetime.now().isoformat()
    }

# Test avec les dernières données
print("Test de prédiction avec données actuelles:")
print("="*50)

result = predict_action(best_model, df.tail(60))  # 60 derniers jours
print(f"Action recommandée: {result['action']}")
print(f"Prix actuel: ${result['current_price']:,.2f}")
print(f"Timestamp: {result['timestamp']}")

---
## Résumé

Ce notebook a:

1. **Récupéré** 3 ans de données BTC depuis Twelve Data API
2. **Créé** des features techniques (RSI, MACD, BB, etc.)
3. **Construit** un environnement de trading Gymnasium
4. **Entraîné** un agent PPO avec reward basé sur Sharpe Ratio
5. **Évalué** les performances sur données de test
6. **Sauvegardé** le modèle pour utilisation dans TraderPro

### Fichiers générés:
- `trading_agent_ppo_final.zip` - Le modèle entraîné
- `training_config_rl.json` - Configuration et métriques
- `rl_backtest_results.png` - Visualisation des résultats

### Prochaines étapes:
1. Placer les fichiers dans `backend/app/ml/models/`
2. Implémenter le service RL dans le backend
3. Créer les endpoints API `/api/rl/action` et `/api/rl/status`